# PCA vs EFA Comparisions

In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.feature_selection import RFE
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from factor_analyzer import FactorAnalyzer

# Load the dataset
file_path = '/Users/hunterberberich/Downloads/cleaned_data_survive.csv'
data = pd.read_csv(file_path)

# Preparing the data
X = data.drop(columns=['date', 'survive'])
y = data['survive']

# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split the data for evaluation
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)



In [4]:
# PCA
pca = PCA(n_components=0.95)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

rfc_pca = RandomForestClassifier(n_estimators=100, random_state=42)
rfc_pca.fit(X_train_pca, y_train)
y_pred_pca = rfc_pca.predict(X_test_pca)

print("PCA Model")
print("Accuracy:", accuracy_score(y_test, y_pred_pca))
print("Classification Report:\n", classification_report(y_test, y_pred_pca))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_pca))



PCA Model
Accuracy: 0.7868068833652008
Classification Report:
               precision    recall  f1-score   support

         0.0       0.13      0.05      0.07       172
         1.0       0.83      0.93      0.88       874

    accuracy                           0.79      1046
   macro avg       0.48      0.49      0.48      1046
weighted avg       0.72      0.79      0.75      1046

Confusion Matrix:
 [[  9 163]
 [ 60 814]]


In [5]:
# RFE
rfc = RandomForestClassifier(n_estimators=100, random_state=42)
rfe = RFE(estimator=rfc, n_features_to_select=10, step=1)
rfe.fit(X_train, y_train)
X_train_rfe = rfe.transform(X_train)
X_test_rfe = rfe.transform(X_test)

rfc_rfe = RandomForestClassifier(n_estimators=100, random_state=42)
rfc_rfe.fit(X_train_rfe, y_train)
y_pred_rfe = rfc_rfe.predict(X_test_rfe)

print("\nRFE Model")
print("Accuracy:", accuracy_score(y_test, y_pred_rfe))
print("Classification Report:\n", classification_report(y_test, y_pred_rfe))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_rfe))




RFE Model
Accuracy: 0.7887189292543021
Classification Report:
               precision    recall  f1-score   support

         0.0       0.14      0.06      0.08       172
         1.0       0.83      0.93      0.88       874

    accuracy                           0.79      1046
   macro avg       0.49      0.50      0.48      1046
weighted avg       0.72      0.79      0.75      1046

Confusion Matrix:
 [[ 10 162]
 [ 59 815]]


In [6]:
# EFA
fa = FactorAnalyzer(n_factors=10, rotation='varimax')
fa.fit(X_train)
X_train_efa = fa.transform(X_train)
X_test_efa = fa.transform(X_test)

rfc_efa = RandomForestClassifier(n_estimators=100, random_state=42)
rfc_efa.fit(X_train_efa, y_train)
y_pred_efa = rfc_efa.predict(X_test_efa)

print("\nEFA Model")
print("Accuracy:", accuracy_score(y_test, y_pred_efa))
print("Classification Report:\n", classification_report(y_test, y_pred_efa))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_efa))




EFA Model
Accuracy: 0.7896749521988528
Classification Report:
               precision    recall  f1-score   support

         0.0       0.15      0.06      0.08       172
         1.0       0.83      0.93      0.88       874

    accuracy                           0.79      1046
   macro avg       0.49      0.50      0.48      1046
weighted avg       0.72      0.79      0.75      1046

Confusion Matrix:
 [[ 10 162]
 [ 58 816]]


In [7]:
# Factor Loadings from EFA
loadings = fa.loadings_
features = X.columns
loadings_df = pd.DataFrame(loadings, index=features, columns=[f'Factor {i+1}' for i in range(loadings.shape[1])])

print("Factor Loadings from EFA:")
print(loadings_df)

most_influential_variables = loadings_df.apply(lambda row: row.abs().nlargest(1).index, axis=1)
print("\nMost Influential Variables in Each Factor:")
print(most_influential_variables)

# Additional Models using EFA features
models_performance = []

lr = LogisticRegression(random_state=42)
lr.fit(X_train_efa, y_train)
y_pred_lr = lr.predict(X_test_efa)
models_performance.append(('Logistic Regression', accuracy_score(y_test, y_pred_lr), classification_report(y_test, y_pred_lr), confusion_matrix(y_test, y_pred_lr)))

rfc = RandomForestClassifier(n_estimators=100, random_state=42)
rfc.fit(X_train_efa, y_train)
y_pred_rfc = rfc.predict(X_test_efa)
models_performance.append(('Random Forest', accuracy_score(y_test, y_pred_rfc), classification_report(y_test, y_pred_rfc), confusion_matrix(y_test, y_pred_rfc)))

gbc = GradientBoostingClassifier(n_estimators=100, random_state=42)
gbc.fit(X_train_efa, y_train)
y_pred_gbc = gbc.predict(X_test_efa)
models_performance.append(('Gradient Boosting', accuracy_score(y_test, y_pred_gbc), classification_report(y_test, y_pred_gbc), confusion_matrix(y_test, y_pred_gbc)))

svm = SVC(random_state=42)
svm.fit(X_train_efa, y_train)
y_pred_svm = svm.predict(X_test_efa)
models_performance.append(('SVM', accuracy_score(y_test, y_pred_svm), classification_report(y_test, y_pred_svm), confusion_matrix(y_test, y_pred_svm)))

for model_name, accuracy, report, cm in models_performance:
    print(f"{model_name} Model")
    print("Accuracy:", accuracy)
    print("Classification Report:\n", report)
    print("Confusion Matrix:\n", cm)
    print("\n")

Factor Loadings from EFA:
                                   Factor 1  Factor 2  Factor 3  Factor 4  \
tempmax                            0.957705 -0.009826  0.018948 -0.202304   
tempmin                            0.959915  0.041289  0.067613  0.056086   
temp                               0.976917  0.008846  0.034195 -0.093894   
feelslikemax                       0.953900 -0.003071  0.025297 -0.185251   
feelslikemin                       0.959501  0.030740  0.064007  0.041668   
feelslike                          0.973346  0.010090  0.036975 -0.093879   
dew                                0.947705  0.128523  0.092608  0.217041   
humidity                           0.121432  0.403565  0.182325  0.736000   
precip                             0.103505  0.663745  0.130795  0.156125   
precipcover                       -0.017725  0.936445  0.221567  0.161547   
snow                              -0.158423  0.032949  0.045024  0.030994   
snowdepth                         -0.372413 -0.035

/Users/hunterberberich/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/hunterberberich/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/hunterberberich/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result)

Logistic Regression Model
Accuracy: 0.8355640535372849
Classification Report:
               precision    recall  f1-score   support

         0.0       0.00      0.00      0.00       172
         1.0       0.84      1.00      0.91       874

    accuracy                           0.84      1046
   macro avg       0.42      0.50      0.46      1046
weighted avg       0.70      0.84      0.76      1046

Confusion Matrix:
 [[  0 172]
 [  0 874]]


Random Forest Model
Accuracy: 0.7896749521988528
Classification Report:
               precision    recall  f1-score   support

         0.0       0.15      0.06      0.08       172
         1.0       0.83      0.93      0.88       874

    accuracy                           0.79      1046
   macro avg       0.49      0.50      0.48      1046
weighted avg       0.72      0.79      0.75      1046

Confusion Matrix:
 [[ 10 162]
 [ 58 816]]


Gradient Boosting Model
Accuracy: 0.8307839388145315
Classification Report:
               precision    re

/Users/hunterberberich/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/hunterberberich/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/hunterberberich/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1469: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result)

In [8]:
features

Index(['tempmax', 'tempmin', 'temp', 'feelslikemax', 'feelslikemin',
       'feelslike', 'dew', 'humidity', 'precip', 'precipcover', 'snow',
       'snowdepth', 'cloudcover', 'uvindex', 'severerisk', 'num_date',
       'preciptype_freezingrain', 'preciptype_freezingrain,snow',
       'preciptype_freezingrain,snow,ice', 'preciptype_none',
       'preciptype_rain', 'preciptype_rain,freezingrain',
       'preciptype_rain,freezingrain,snow', 'preciptype_rain,snow',
       'preciptype_rain,snow,ice', 'preciptype_snow'],
      dtype='object')